# Fiche 5 — Référence, recherche d'hyperparamètres et journal MLflow

| | |
|---|---|
| 🎯 **Objectif** | Comparer un modèle à une référence naïve, chercher des hyperparamètres de façon systématique et tracer chaque essai dans MLflow |
| ⏱️ **Durée** | 60 min |
| 🧰 **Prérequis** | Fiche 4 |
| 🔗 **Dans le projet** | Étape 2 (optimisation, MLflow, captures) |

> **L'idée clé.** Un score ne veut rien dire seul : on le compare à une **référence naïve**. On cherche les réglages **systématiquement** (grille ou tirages aléatoires), sur les mêmes plis, et on **note tout** dans MLflow, pour pouvoir expliquer chaque choix.

In [1]:
import tempfile

from agrireco import io, prepare  # en premier : le package coupe un message d'aide de MLflow

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor

_, fao_dir = io.download()
# ⚠️ Cette cellule utilise la fusion du projet (agrireco.prepare) : construis d'abord la tienne (fiches 1 et 2)
fao, _ = prepare.build_fao_table(*prepare.load_fao_raw(fao_dir))
df = prepare.add_model_features(fao)
X = pd.get_dummies(df[["crop", "avg_temp", "rainfall_mm", "log10_pesticides_t"]], dtype=float)
y = np.log(df["yield_t_ha"])
plis = list(GroupKFold(5).split(X, y, groups=df["country"]))  # LES mêmes plis pour tout le monde

## 1. La référence : un modèle qui ne regarde pas les conditions

In [2]:
def rmse_reference(plis):
    """Prédit, pour chaque culture, la médiane du log du rendement à l'entraînement."""
    scores = []
    for tr, va in plis:
        mediane = y.iloc[tr].groupby(df["crop"].iloc[tr]).median()
        pred = df["crop"].iloc[va].map(mediane)
        scores.append(np.sqrt(np.mean((y.iloc[va] - pred) ** 2)))
    return np.array(scores)

ref = rmse_reference(plis)
arbre = -cross_val_score(DecisionTreeRegressor(random_state=0), X, y, cv=plis, scoring="neg_root_mean_squared_error")
print(f"référence (médiane de la culture) : {ref.mean():.3f}")
print(f"arbre de décision, réglages par défaut : {arbre.mean():.3f}")

référence (médiane de la culture) : 0.719
arbre de décision, réglages par défaut : 0.875


## ⚠️ Piège : un modèle « sophistiqué » peut faire pire que rien
L'arbre par défaut pousse jusqu'à une feuille par ligne : il apprend chaque pays par cœur et fait **pire que la
référence** sur des pays nouveaux. Sans la référence, on aurait pu croire qu'une erreur de cet ordre était normale.

## 2. Chercher les réglages de façon systématique
`RandomizedSearchCV` tire des combinaisons au hasard dans des plages qu'on choisit (et qu'on doit savoir justifier),
et évalue chacune sur les mêmes plis.

In [3]:
plages = {
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_leaf": [1, 5, 20, 50, 100, 200],  # taille minimale d'une feuille : le frein à la mémorisation
}
recherche = RandomizedSearchCV(DecisionTreeRegressor(random_state=0), plages, n_iter=12, cv=plis,
                               scoring="neg_root_mean_squared_error", random_state=0)
recherche.fit(X, y)
resultats = pd.DataFrame(recherche.cv_results_)[["param_max_depth", "param_min_samples_leaf", "mean_test_score", "std_test_score"]]
resultats.assign(rmse=-resultats["mean_test_score"]).sort_values("rmse").head(5)

,param_max_depth,param_min_samples_leaf,mean_test_score,std_test_score,rmse
8,8,200,-0.651904,0.085027,0.651904
11,8,100,-0.677105,0.091632,0.677105
9,12,100,-0.687275,0.088121,0.687275
1,None,100,-0.689315,0.091064,0.689315
2,8,5,-0.697486,0.097909,0.697486


L'arbre réglé passe **sous la référence** : il apporte enfin quelque chose, alors que l'arbre par défaut faisait pire.
C'est la comparaison « référence contre modèle optimisé » attendue. Remarque aussi que le meilleur
`min_samples_leaf` est **la plus grande valeur testée** : l'optimum est peut-être au-delà. Quand le meilleur
réglage est au bord de la plage, on élargit la plage.

## 3. Tout noter dans MLflow
Un **run** MLflow = un essai : ses paramètres, ses métriques, ses tags (une description), éventuellement le modèle.
On regroupe les runs dans une **expérience**. Ici, on écrit dans un dossier temporaire pour ne pas mélanger avec
le suivi du projet.

In [4]:
dossier = tempfile.mkdtemp()
mlflow.set_tracking_uri(f"sqlite:///{dossier}/mlflow.db")
mlflow.set_experiment("fiche-5-arbre")
for _, ligne in pd.DataFrame(recherche.cv_results_).iterrows():
    with mlflow.start_run(run_name=f"arbre-{ligne['param_max_depth']}-{ligne['param_min_samples_leaf']}"):
        mlflow.log_params({"max_depth": ligne["param_max_depth"], "min_samples_leaf": ligne["param_min_samples_leaf"], "seed": 0})
        mlflow.log_metrics({"cv_rmse_log_mean": -ligne["mean_test_score"], "cv_rmse_log_std": ligne["std_test_score"]})
        mlflow.set_tag("mlflow.note.content", "arbre de décision, CV groupée par pays (fiche 5)")
runs = mlflow.search_runs()
runs[["tags.mlflow.runName", "params.max_depth", "params.min_samples_leaf", "metrics.cv_rmse_log_mean"]].sort_values("metrics.cv_rmse_log_mean").head()

2026/10/25 03:31:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/25 03:31:44 INFO mlflow.store.db.utils: Updating database tables


2026/10/25 03:31:44 INFO mlflow.tracking.fluent: Experiment with name 'fiche-5-arbre' does not exist. Creating a new experiment.


,tags.mlflow.runName,params.max_depth,params.min_samples_leaf,metrics.cv_rmse_log_mean
3,arbre-8-200,8,200,0.651904
0,arbre-8-100,8,100,0.677105
2,arbre-12-100,12,100,0.687275
10,arbre-None-100,None,100,0.689315
9,arbre-8-5,8,5,0.697486


## 4. Sauvegarder le modèle retenu, et vérifier qu'il se recharge

In [5]:
import joblib

modele = DecisionTreeRegressor(random_state=0, **recherche.best_params_).fit(X, y)
chemin = f"{dossier}/modele.joblib"
joblib.dump(modele, chemin)
recharge = joblib.load(chemin)
print("prédictions identiques après rechargement :", np.allclose(modele.predict(X), recharge.predict(X)))

prédictions identiques après rechargement : True


Un modèle picklé ne se recharge de façon garantie qu'avec **la même version** de scikit-learn : on fige les
versions (fichier de verrouillage) et on note la version dans les métadonnées du modèle.

## 5. Ce que le modèle utilise : l'importance par permutation
On mélange une variable et on regarde de combien l'erreur augmente. Attention à ce que ce chiffre **ne dit pas**.

In [6]:
from sklearn.inspection import permutation_importance

tr, va = plis[0]
m = DecisionTreeRegressor(random_state=0, **recherche.best_params_).fit(X.iloc[tr], y.iloc[tr])
imp = permutation_importance(m, X.iloc[va], y.iloc[va], n_repeats=5, random_state=0,
                             scoring="neg_root_mean_squared_error")
pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).head(6).round(3)

crop_Soybeans       0.352
crop_Sorghum        0.280
crop_Potatoes       0.263
crop_Maize          0.262
crop_Wheat          0.246
crop_Rice, paddy    0.130
dtype: float64

- Ici, les colonnes de **culture** arrivent en tête : la culture fixe l'ordre de grandeur du rendement. Note que
  l'encodage one-hot découpe la variable « culture » en dix colonnes, dont l'importance se répartit.
- Une importance mesure ce que **ce modèle** utilise, sur **ces données** : pas un effet causal, ni son sens (plus
  de pesticides = plus ou moins de rendement ?).
- Avec des variables corrélées (pluie, température, pesticides varient ensemble d'un pays à l'autre), l'importance
  se partage de façon arbitraire entre elles.
- Les pesticides sont des tonnes **par pays** : leur importance reflète d'abord la taille et l'équipement agricole
  du pays.

Pour voir ces runs dans l'interface : `uv run mlflow ui --backend-store-uri sqlite:///<dossier>/mlflow.db`
(le chemin est dans la variable `dossier`). Dans l'onglet *Runs*, ajoute les colonnes de métriques
(*Columns*), trie par `cv_rmse_log_mean`, et sélectionne plusieurs runs pour *Compare*.

### ✍️ À toi de jouer

Ajoute à la recherche l'hyperparamètre `max_features` (valeurs `[0.3, 0.6, 1.0]`), relance avec `n_iter=20` et enregistre les essais dans une **nouvelle** expérience MLflow `fiche-5-arbre-v2`. Le meilleur score a-t-il beaucoup changé ? Quel hyperparamètre semble décisif ?

In [7]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
plages2 = {**plages, "max_features": [0.3, 0.6, 1.0]}
r2 = RandomizedSearchCV(DecisionTreeRegressor(random_state=0), plages2, n_iter=20, cv=plis,
                        scoring="neg_root_mean_squared_error", random_state=0).fit(X, y)
mlflow.set_experiment("fiche-5-arbre-v2")
for _, l in pd.DataFrame(r2.cv_results_).iterrows():
    with mlflow.start_run():
        mlflow.log_params({k.removeprefix("param_"): l[k] for k in l.index if k.startswith("param_")})
        mlflow.log_metric("cv_rmse_log_mean", -l["mean_test_score"])
print(r2.best_params_, round(-r2.best_score_, 3))
```

**Pourquoi ?** regarde les meilleurs essais : `min_samples_leaf` élevé revient presque toujours. C'est lui qui empêche l'arbre de mémoriser les pays. Savoir dire **quel hyperparamètre a compté et pourquoi** est exactement ce qu'on attend à l'oral. Le meilleur score d'une recherche est un peu optimiste (on a choisi le meilleur de N essais) : d'où un jeu de test gardé de côté jusqu'à la fin.
</details>

## ✅ À retenir

1. Toujours une référence naïve : un score ne s'interprète que par comparaison.
2. Recherche systématique (grille ou aléatoire), plages justifiées, mêmes plis pour tous les essais.
3. Un run MLflow = paramètres + métriques + tags ; une expérience = un groupe de runs comparables.
4. On note la graine et une description : un run doit pouvoir être compris et rejoué.
5. Le meilleur score de la recherche est optimiste : le test final, mis de côté, le corrige.
6. Meilleur réglage au bord de la plage → on élargit la plage.